# One filing, one stage at a time

Use the Phase 2 architecture in `project.md` and the approved design with **AMD's 2022 10-K**.
Change `DOC_NAME` to another exact manifest name to select one different filing.
This is a document-scoped learning experiment, not the 53-filing acceptance benchmark.

From the repository root run `uv sync --locked --group notebook`. Open this notebook in
VS Code/Jupyter and select the repository's `.venv` Python kernel. Start at the top.
All network steps ship with `run=False`: **Run All performs local work only**.
To call a stage yourself, change its example to `run=True`, execute it, then inspect
the result before moving on. Never print `settings`, environment variables, or keys.
Clear all outputs before sharing or committing the notebook.

```text
original PDF + manifest (one filing, verified hashes)
    -> isolated extracted pages -> pinned tokenizer -> fixed chunks + source spans
    -> shared embedding cache <- one manual Nebius batch at a time [paid]
    -> isolated Chroma index + integrity manifest [cache only]

editable question + explicit document filter
    -> Claude question admission [token-count endpoint]
    -> Nebius query embedding [paid on cache miss]
    -> filtered dense retrieval -> ranked candidates [local]
    -> pack exact evidence slices [token-count endpoint]
    -> grounded prompt + schema [local] -> padded admission [token-count endpoint]
    -> Claude structured draft [paid] -> citation/calculation validation [local]
    -> answer + original PDF pages + local walkthrough receipt

all paid calls -> existing shared budget ledger + fresh verified rates
```

Small batches reduce request size; a single request can still time out. No timeout
increase or timeout retry is introduced. A timeout retains its uncertain billing
reservation. Inspect the error and ledger before explicitly trying another call.
The shared $5 ceiling, $4 working allowance, $1 reserve, and $0.75 Phase 2 sublimit
include previous activity. Do not delete, replace, relocate, or reinitialize budget files.

## 1. Imports and repository location — local

Cells use real project functions. The small helpers below only separate stages and
guard interactive state. Run setup again (or restart the kernel) after editing
configuration or stage definitions. Changes to production modules require a kernel restart.

In [ ]:
from collections import Counter
from pathlib import Path
from pprint import pprint
from time import monotonic
from uuid import uuid4

import httpx

from finsight.application import BaselineApplicationService
from finsight.chunking.fixed import chunk_pages
from finsight.chunking.tokenizer import LocalOffsetTokenizer, TokenizerArtifact
from finsight.citations.validation import validate_draft
from finsight.config import Settings
from finsight.context.packing import pack_evidence
from finsight.contracts import BaselineConfig, QueryRequest
from finsight.contracts.answers import UsageMetrics
from finsight.contracts.base import fingerprint
from finsight.contracts.ingestion import CorpusManifest, ExtractionConfig
from finsight.generation.claude import ClaudeGenerator, create_chat
from finsight.generation.counting import ClaudeTokenCounter, admit_input_tokens, validate_question
from finsight.generation.prompts import build_prompt
from finsight.indexing.builder import build_index, open_index
from finsight.indexing.catalog import create_catalog
from finsight.indexing.embedding_cache import EmbeddingCache
from finsight.indexing.nebius import CachedNebiusEmbeddings
from finsight.ingestion.cache import extract_cached, write_json_atomic
from finsight.ingestion.fingerprints import sha256_file, source_fingerprint
from finsight.ingestion.manifest import safe_pdf_path
from finsight.ingestion.pages import CachedPageRepository
from finsight.observability.dispatch import GuardedDispatcher
from finsight.observability.tracing import JsonlTraceRecorder
from finsight.runtime.errors import FinsightError
from finsight.runtime.factory import preflight, spending_setup
from finsight.runtime.ports import CallContext

# Re-running setup closes resources from the previous walkthrough first.
if callable(globals().get("close_resources")):
    globals()["close_resources"](run=True)
ROOT = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "project.md").is_file() and (p / "pyproject.toml").is_file()
)
print("Repository:", ROOT)

## 2. Select exactly one filing — local

The original manifest is read only. No benchmark questions, gold pages, or split files
are loaded. This selector is your explicit document filter, not an inferred answer hint.

In [ ]:
DOC_NAME = "AMD_2022_10K"
CHUNK_TOKENS = 400
OVERLAP_TOKENS = 60
CANDIDATE_COUNT = 15
TOP_K = 5
BATCH_SIZE = 4  # One explicit call sends at most this many unique missing inputs (1..16).

parent_corpus = CorpusManifest.model_validate_json(
    (ROOT / "data/manifests/financebench-selected.json").read_text(encoding="utf-8")
)
matches = [d for d in parent_corpus.documents if d.doc_name == DOC_NAME]
if len(matches) != 1:
    raise ValueError("Choose one exact doc_name from parent_corpus.documents")
document = matches[0]
scoped_corpus = CorpusManifest(
    documents=(document,),
    metadata_sha256=parent_corpus.metadata_sha256,
    source_fingerprint=source_fingerprint([[document.doc_name, document.sha256]]),
    diagnostics=tuple(d for d in parent_corpus.diagnostics if DOC_NAME in d.doc_names),
)
pprint(document.model_dump(mode="json"))
print(
    "Other filings for this company:",
    [d.doc_name for d in parent_corpus.documents if d.company == document.company],
)

## 3. Isolated artifacts and bounded configuration — local

Pages, the scoped manifest, indexes, and traces live in ignored notebook directories.
The text-keyed embedding cache is shared, so identical inputs can reuse paid vectors.
The budget database and independent identity authority are the **existing** project files.
Configuration validation keeps the approved caps. Editing chunk sizes changes the scoped
config fingerprint and index path; rerun setup and downstream stages afterward.

In [ ]:
config = BaselineConfig(
    source_fingerprint=scoped_corpus.source_fingerprint,
    chunk_tokens=CHUNK_TOKENS,
    overlap_tokens=OVERLAP_TOKENS,
    candidate_count=CANDIDATE_COUNT,
    top_k=TOP_K,
)
# Short path labels; full identities remain validated in manifests.
document_label = document.document_id[-16:]
lab_root = ROOT / "data/cache/notebook" / document_label
settings = Settings(
    _env_file=ROOT / ".env",
    source_pdf_dir=ROOT / "source-data/pdfs",
    manifest_path=lab_root / "manifest.json",
    page_cache_root=lab_root / "pages",
    tokenizer_dir=ROOT / "data/cache/tokenizer/qwen3-embedding-8b",
    embedding_cache_root=ROOT / "data/cache/embeddings",
    index_root=ROOT / "data/indexes/notebook" / document_label,
    rates_path=ROOT / "data/cache/budget/rates.json",
    ledger_path=ROOT / "data/cache/budget/testing.sqlite3",
    budget_anchor_path=ROOT / "results/testing-budget-identity.json",
    trace_root=ROOT / "results/traces/notebook" / document_label,
)
write_json_atomic(settings.manifest_path, scoped_corpus.model_dump(mode="json"))
index_path = settings.index_root / config.fingerprint()[:16]
print("Scoped source:", config.source_fingerprint)
print("Config:", config.fingerprint())
print("Index:", index_path)
print(
    "Key presence only:",
    dict(nebius=bool(settings.nebius_api_key), claude=bool(settings.anthropic_api_key)),
)

## 4. Extract the original PDF — local

`extract_cached` validates the original hash and uses the Phase 1 extraction rules.
It writes a separate notebook page cache. Text extraction preserves source offsets;
it does not reconstruct financial tables or add OCR. Inspect diagnostics and compare
the sample with the PDF. `page_index` is zero based; PDF viewer page is index + 1.

In [ ]:
pdf_path = safe_pdf_path(settings.source_pdf_dir, document.local_path)
assert sha256_file(pdf_path) == document.sha256
extraction = ExtractionConfig()
pages_tuple = extract_cached(
    settings.source_pdf_dir, document, extraction, settings.page_cache_root
)
assert len(pages_tuple) == document.page_count
assert not any(p.extraction_status == "error" for p in pages_tuple)
page_repo = CachedPageRepository(scoped_corpus, settings.source_pdf_dir, settings.page_cache_root)
page_repo.validate(document.document_id)
print("PDF:", pdf_path)
print(
    "Pages:", len(pages_tuple), "Status:", dict(Counter(p.extraction_status for p in pages_tuple))
)
print("Pages with table diagnostics:", sum(bool(p.table_diagnostics) for p in pages_tuple))

In [ ]:
PAGE_INDEX = 0  # Edit to inspect any original extracted page.
page = page_repo.get(document.document_id, PAGE_INDEX)
print("PDF viewer page:", page.page_index + 1, "Diagnostics:", page.table_diagnostics)
print(page.text[:2000])

## 5. Pinned tokenizer and source-preserving chunks — local

Load the already-prepared tokenizer artifacts only; no model weights or downloads.
Fixed page-local windows use at most 400 tokens with 60-token overlap. Inspect the
exact character interval behind each chunk; no gold evidence participates.

In [ ]:
artifact = TokenizerArtifact.load(settings.tokenizer_dir)
if artifact.model != config.tokenizer_model or artifact.revision != config.tokenizer_revision:
    raise FinsightError("tokenizer_integrity")
tokenizer = LocalOffsetTokenizer(artifact)
print("Tokenizer:", artifact.model, artifact.revision)
chunks = chunk_pages(document, pages_tuple, config, tokenizer)
assert chunks and all(c.token_count <= config.chunk_tokens for c in chunks)
for c in chunks:
    assert (
        "".join(
            page_repo.get(c.document_id, s.page_index).text[s.char_start : s.char_end]
            for s in c.source_spans
        )
        == c.content
    )
catalog = create_catalog(scoped_corpus, page_repo, config, tokenizer)
assert catalog.chunks == chunks
print("Chunks:", len(chunks), "Max tokens:", max(c.token_count for c in chunks))
print("Catalog fingerprint:", catalog.payload_fingerprint)

In [ ]:
CHUNK_INDEX = 0
pprint(chunks[CHUNK_INDEX].model_dump(mode="json"))

## 6. Existing budget, fresh rates, cache progress — local

Preflight makes zero network calls. It reserves no money and reports a conservative
upper estimate for this filing plus four queries, including eligible retries. Actual
reservation happens atomically at dispatch. Missing/corrupt budget identity or rates
older than 24 hours stop paid stages. Reverify official rates before updating the
rate book; never change timestamps merely to bypass expiry. See the repository README.

In [ ]:
cache = EmbeddingCache(settings.embedding_cache_root, config)
rates = ledger = report = None
try:
    rates, ledger = spending_setup(settings, config)
    report = preflight(settings, config)
    pprint(report)
except FinsightError as exc:
    print(
        "Paid stages blocked:",
        exc.code,
        "— inspect the existing budget/rate setup; do not reset it.",
    )


def cache_progress():
    cached = sum(cache.get(cache.key(c.content, "document")) is not None for c in chunks)
    return dict(total_chunks=len(chunks), cached_chunks=cached, missing_chunks=len(chunks) - cached)


pprint(cache_progress())

## 7. Clients and interactive state — constructors only, no requests

Each manual stage gets a fresh deadline so pauses between cells do not consume its
45-second limit. Each physical attempt keeps its 20-second limit. This manual
walkthrough does not measure the application's whole-request latency. An optional
final service call below uses the real shared 45-second end-to-end deadline.

In [ ]:
if callable(globals().get("close_resources")):
    globals()["close_resources"](run=True)
backend = embedding_http = chat = counter = generator = dispatcher = None
retriever = index_manifest = None
question_admission_key = query_binding = retrieval_key = packet_key = prompt_key = draft_key = None
candidates = packet = prompt = draft = answer = None
stage_runs = []
setup_fingerprint = config.fingerprint()
walkthrough_id = uuid4().hex


def stage_context(purpose, seconds=None):
    identity = f"{walkthrough_id}-{purpose}-{uuid4().hex[:8]}"
    stage_runs.append(dict(run_id=identity, purpose=purpose))
    return CallContext(
        identity,
        2,
        monotonic() + (config.deadline_s if seconds is None else seconds),
        clock=monotonic,
    )


def require_setup():
    if config.fingerprint() != setup_fingerprint:
        raise FinsightError("notebook_stale_stage", "Configuration changed; restart setup")


if rates is not None and ledger is not None:
    dispatcher = GuardedDispatcher(ledger, rates, config=config)
    if settings.nebius_api_key:
        embedding_http = httpx.Client(
            transport=httpx.HTTPTransport(retries=0),
            timeout=config.attempt_timeout_s,
            follow_redirects=False,
        )
        backend = CachedNebiusEmbeddings(
            embedding_http, settings.nebius_api_key, tokenizer, cache, config, dispatcher
        )
    if settings.anthropic_api_key:
        chat = create_chat(config, settings.anthropic_api_key)
        counter = ClaudeTokenCounter(
            chat._client, config.generation_model, attempt_timeout_s=config.attempt_timeout_s
        )
        generator = ClaudeGenerator(config, dispatcher, counter, chat=chat)
print("Constructed:", dict(embeddings=backend is not None, generation=generator is not None))


def require_paid_stage(provider):
    require_setup()
    fresh_rates, _ = spending_setup(settings, config)
    if dispatcher is None or fresh_rates.fingerprint() != dispatcher.rates.fingerprint():
        raise FinsightError("notebook_stale_stage", "Rates changed; rerun setup")
    if provider == "nebius" and backend is None or provider == "anthropic" and generator is None:
        raise FinsightError("credentials_missing")
    if preflight(settings, config)["status"] != "ready":
        raise FinsightError("budget_exceeded")

## 8. Embed just the next small batch — paid on cache miss, explicit

Change **only this call** to `run=True`, execute it, inspect progress, then repeat
the cell until missing chunks reaches zero. Each invocation selects at most four
unique missing texts by default; no outer loop runs through the filing. Use
`BATCH_SIZE = 1` for smaller requests. Cached vectors survive a kernel restart.

A provider timeout stops the call without automatic timeout retry. The production
dispatcher may retry one eligible 429/502/503/504 response within the same deadline
and budget. Unknown billing stays reserved. Repeating a failed cell is a new explicit
attempt and may spend again; inspect the ledger first.

In [ ]:
def embed_next_batch(*, run=False):
    if not run:
        print("Embedding OFF. Set run=True to send one small missing batch.")
        return None
    require_paid_stage("nebius")
    if type(BATCH_SIZE) is not int or not 1 <= BATCH_SIZE <= 16:
        raise ValueError("BATCH_SIZE must be an integer from 1 to 16")
    pending = {}
    for c in chunks:
        key = cache.key(c.content, "document")
        if cache.get(key) is None:
            pending.setdefault(key, c.content)
            if len(pending) == BATCH_SIZE:
                break
    if not pending:
        print("All chunk embeddings are cached; build the local index next.")
        return cache_progress()
    try:
        backend.for_run(stage_context("document-embedding")).embed_documents(list(pending.values()))
    finally:
        pprint(cache_progress())
        pprint(ledger.snapshot().model_dump(mode="json"))
    return cache_progress()


embed_next_batch(run=False)

## 9. Build/reopen the scoped Chroma index — cache only, no network

This adapter reads vectors exclusively from the validated embedding cache. Indexing
cannot silently purchase missing embeddings. `build_index` verifies the catalog,
Chroma rows, canonical float32 vector references, configuration, and source identities.
It publishes a complete manifest only when every chunk is present. Reruns reopen a
compatible completed index. The full-corpus index is separate and untouched.

In [ ]:
class CachedDocumentsOnly:
    def for_run(self, context):
        context.remaining()
        return self

    def embed_documents(self, texts):
        records = [cache.get(cache.key(t, "document")) for t in texts]
        if any(r is None for r in records):
            raise FinsightError("notebook_embeddings_missing")
        return [list(r.vector) for r in records]

    def embed_query(self, text):
        raise FinsightError("notebook_query_not_embedded")


def build_cached_index():
    global index_manifest
    require_setup()
    index_manifest = None
    if cache_progress()["missing_chunks"]:
        print("Embed the missing chunks explicitly first; index stage made no provider calls.")
        return None
    index_manifest = build_index(
        scoped_corpus,
        page_repo,
        config,
        CachedDocumentsOnly(),
        index_path,
        stage_context("cache-only-index", settings.index_job_timeout_s),
        tokenizer=tokenizer,
    )
    print("Complete:", index_manifest.complete, "Chunks:", index_manifest.chunk_count)
    print("Index fingerprint:", index_manifest.fingerprint())
    return index_manifest


index_manifest = build_cached_index()

## 10. Write your question and admit its size — explicit token-count endpoint

Edit the question freely. The scope is this one filing, with manifest-derived
company/period filters. Changing the question requires rerunning admission, query
embedding, retrieval, packing, prompt, generation, and validation. Guards reject
results bound to an earlier request. Token counting contacts Claude but does not
perform a generation; it is still off by default and has no local fallback.

In [ ]:
request = QueryRequest(
    question=f"What was {document.company}'s revenue in {document.fiscal_period}?",
    company=document.company,
    period=document.fiscal_period,
    document_ids=(document.document_id,),
    top_k=config.top_k,
)


def current_request_key():
    require_setup()
    return fingerprint(dict(config=config.fingerprint(), request=request.model_dump(mode="json")))


def require_binding(binding):
    if binding != current_request_key():
        raise FinsightError(
            "notebook_stale_stage", "Rerun the upstream stages for the current request"
        )


def clear_after(stage):
    # Clear outputs before starting, so a failed rerun never displays an earlier success.
    global \
        query_binding, \
        candidates, \
        retrieval_key, \
        packet, \
        packet_key, \
        prompt, \
        prompt_key, \
        draft, \
        draft_key, \
        answer
    if stage == "question":
        query_binding = None
    if stage in ("question", "embedding", "retrieval"):
        candidates = retrieval_key = None
    if stage in ("question", "embedding", "retrieval", "packing"):
        packet = packet_key = None
    if stage in ("question", "embedding", "retrieval", "packing", "prompt"):
        prompt = prompt_key = None
    draft = draft_key = answer = None


def admit_question(*, run=False):
    global question_admission_key
    if not run:
        print("Question admission OFF. Set run=True to count with Claude.")
        return None
    question_admission_key = None
    clear_after("question")
    if counter is None:
        raise FinsightError("credentials_missing")
    count = validate_question(
        request.question, counter, stage_context("question-admission"), config
    )
    question_admission_key = current_request_key()
    print("Question tokens:", count, "Limit:", config.question_tokens)
    return count


pprint(request.model_dump(mode="json"))
admit_question(run=False)

## 11. Embed the question separately — paid on cache miss, explicit

The same backend adds the configured retrieval instruction exactly once. A cached
identical query costs nothing. Its vector is bound to the current request/configuration.
The next stage uses this vector directly, avoiding another query embedding purchase.

In [ ]:
def embed_question(*, run=False):
    global query_binding
    if not run:
        print("Query embedding OFF. Set run=True after question admission.")
        return None
    query_binding = None
    clear_after("embedding")
    require_binding(question_admission_key)
    require_paid_stage("nebius")
    vector = backend.for_run(stage_context("query-embedding")).embed_query(request.question)
    query_binding = (current_request_key(), vector, request.question)
    print("Query dimension:", len(vector))
    return dict(request_fingerprint=query_binding[0], dimension=len(vector))


embed_question(run=False)

## 12. Retrieve ranked evidence — local

Open and validate the scoped index, then use the production `DenseRetriever` with
the already-created query vector. Inspect scores, ranks, document IDs, page indices,
and the original text. Retrieval score is not proof that a passage answers the question.

In [ ]:
class KnownQueryOnly:
    def __init__(self, binding):
        self.binding = binding

    def for_run(self, context):
        context.remaining()
        return self

    def embed_query(self, text):
        require_binding(self.binding[0])
        if text != self.binding[2]:
            raise FinsightError("notebook_stale_stage")
        return list(self.binding[1])

    def embed_documents(self, texts):
        raise FinsightError("notebook_embeddings_missing")


def retrieve_evidence():
    global retriever, candidates, retrieval_key
    clear_after("retrieval")
    if query_binding is None or index_manifest is None:
        print("Complete the index and explicit query embedding first.")
        return None
    require_binding(query_binding[0])
    if retriever is not None:
        retriever.store.client.close()
        retriever = None
    retriever = open_index(
        index_path,
        config,
        scoped_corpus,
        pages=page_repo,
        embeddings=KnownQueryOnly(query_binding),
        tokenizer=tokenizer,
    )
    candidates = retriever.retrieve(request, stage_context("retrieval"))
    retrieval_key = current_request_key()
    pprint(
        [
            dict(
                rank=c.rank,
                score=c.score,
                pdf_page=c.chunk.page_start + 1,
                chunk_id=c.chunk.chunk_id,
                text=c.chunk.content[:350],
            )
            for c in candidates
        ]
    )
    return candidates


candidates = retrieve_evidence()

## 13. Pack retained context — explicit token-count endpoint

`pack_evidence` selects exact page substrings within the 3,500-token context cap.
It handles overlap, budget clipping, and table diagnostics. Inspect retained slices
and dropped IDs; only retained evidence can support citations in the answer.

In [ ]:
def pack_context(*, run=False):
    global packet, packet_key
    if not run:
        print("Packing OFF. Set run=True after retrieval (Claude token counting).")
        return None
    clear_after("packing")
    require_binding(retrieval_key)
    if candidates is None or counter is None:
        raise FinsightError("notebook_stage_missing")
    packet = pack_evidence(
        candidates, page_repo, request, counter, stage_context("packing"), config
    )
    packet_key = current_request_key()
    pprint(packet.model_dump(mode="json"))
    return packet


pack_context(run=False)

## 14. Inspect the grounded prompt and schema — local

This is the production prompt, not a notebook-specific answer shortcut. Source text
is untrusted data. Claims must refer to retained evidence IDs and exact quotes.
An empty/unsuitable packet can lead to an explicit insufficient-evidence result.

In [ ]:
def make_prompt():
    global prompt, prompt_key
    clear_after("prompt")
    if packet is None:
        print("Pack evidence first.")
        return None
    require_binding(packet_key)
    prompt = build_prompt(request, packet)
    prompt_key = current_request_key()
    print("Prompt fingerprint:", prompt.fingerprint)
    print("SYSTEM:\n", prompt.system)
    print("USER (first 2500 characters):\n", prompt.user[:2500])
    print("Schema fields:", list(prompt.schema.get("properties", {})))
    return prompt


prompt = make_prompt()

## 15. Count the complete prompt with padding — explicit token-count endpoint

Count system text, user text, and output schema; add max(512, 10%) padding and enforce
the 6,000-input-token cap. Generation repeats this admission through its cached counter.
Do not substitute character estimates for the provider token-count endpoint.

In [ ]:
def count_full_prompt(*, run=False):
    if not run:
        print("Full prompt admission OFF. Set run=True to count with Claude.")
        return None
    require_binding(prompt_key)
    if prompt is None or counter is None:
        raise FinsightError("notebook_stage_missing")
    estimate = counter.count_prompt(
        prompt.system, prompt.user, prompt.schema, stage_context("prompt-admission")
    )
    padded = admit_input_tokens(estimate, config)
    print("Estimate:", estimate, "Padded:", padded, "Limit:", config.input_tokens)
    return dict(estimate=estimate, padded=padded)


count_full_prompt(run=False)

## 16. Generate one structured draft — paid, explicit

This is the step that purchases a Claude answer. Review the budget snapshot and
prompt first. Usage is settled before output parsing/validation; a malformed draft
can still cost money. No repair generation or model fallback runs. Rerunning a
successful generation can spend again even when all embeddings are cached.

In [ ]:
def generate_draft(*, run=False):
    global draft, draft_key, answer
    if not run:
        print("Generation OFF. Set run=True to purchase one structured draft.")
        return None
    draft = draft_key = answer = None
    require_binding(prompt_key)
    require_paid_stage("anthropic")
    draft = generator.generate(prompt, stage_context("generation"))
    draft_key = (current_request_key(), prompt.fingerprint)
    pprint(draft.model_dump(mode="json"))
    pprint(ledger.snapshot().model_dump(mode="json"))
    return draft


generate_draft(run=False)

## 17. Validate claims, quotes, and calculations — local

Run the production validator against retained slices and the unchanged original pages.
Open each cited PDF page yourself and check fiscal period, row/column association,
units, and whether the cited facts support the answer. Exact quote containment and
arithmetic validation do not establish semantic faithfulness.
Usage below covers paid calls in this kernel setup; the ledger covers all prior activity.

In [ ]:
def walkthrough_usage():
    attempts = (
        tuple(a for a in dispatcher.attempts if a.purpose != "free_token_count")
        if dispatcher
        else ()
    )

    def total(field):
        values = [getattr(a.usage, field, None) for a in attempts]
        return None if any(v is None for v in values) else sum(values)

    return UsageMetrics(
        model_calls=sum(a.provider == "anthropic" for a in attempts),
        embeddings=sum(a.input_count for a in attempts if a.provider == "nebius"),
        input_tokens=total("input_tokens"),
        output_tokens=total("output_tokens"),
        cost_usd=total("cost_usd"),
    )


def validate_answer():
    global answer
    answer = None
    if draft is None:
        print("Generate a draft first.")
        return None
    require_binding(draft_key[0])
    if draft_key[1] != prompt.fingerprint:
        raise FinsightError("notebook_stale_stage")
    answer = validate_draft(draft, packet, page_repo, walkthrough_usage(), request=request)
    pprint(answer.model_dump(mode="json"))
    return answer


answer = validate_answer()

## 18. Save an inspectable manual receipt — local, explicit

This records separate stage identities and current bindings, including stale bindings
if you edited a request. It is not a single-deadline `RunTrace` or benchmark score.
Receipts contain question/source/answer text: keep them local. Paid attempts and the
budget snapshot are useful when investigating a failed call. Never remove budget files
to recover from an error. Save before restarting if you want to retain interactive results.

In [ ]:
def save_walkthrough(*, run=False):
    if not run:
        print("Receipt save OFF. Set run=True to save current state locally.")
        return None
    payload = dict(
        kind="manual_single_document_walkthrough",
        walkthrough_id=walkthrough_id,
        stages=stage_runs,
        config=config.model_dump(mode="json"),
        source_fingerprint=scoped_corpus.source_fingerprint,
        request=request.model_dump(mode="json"),
        request_key=current_request_key(),
        bindings=dict(
            admission=question_admission_key,
            query=query_binding[0] if query_binding else None,
            retrieval=retrieval_key,
            packet=packet_key,
            prompt=prompt_key,
            draft=draft_key,
        ),
        index_fingerprint=index_manifest.fingerprint() if index_manifest else None,
        retrieved=[c.model_dump(mode="json") for c in candidates or ()],
        packet=packet.model_dump(mode="json") if packet else None,
        prompt=prompt.model_dump(mode="json") if prompt else None,
        draft=draft.model_dump(mode="json") if draft else None,
        answer=answer.model_dump(mode="json") if answer else None,
        attempts=[a.model_dump(mode="json") for a in dispatcher.attempts] if dispatcher else [],
        counting=list(getattr(counter, "records", ())),
        budget=ledger.snapshot().model_dump(mode="json") if ledger else None,
    )
    path = settings.trace_root / f"{walkthrough_id}-manual.json"
    write_json_atomic(path, payload)
    print("Saved:", path)
    return path


save_walkthrough(run=False)

## 19. Optional: call the real application service — additional paid generation

After understanding the individual stages, this optional cell composes the same
`BaselineApplicationService` used by the CLI/UI architecture, with the scoped index.
It starts a new 45-second whole-request deadline and writes a standard trace. It can
reuse embeddings but **purchases another generation**; leave it off unless you want
that comparison. It does not run the full-corpus development/holdout campaign.

In [ ]:
def run_shared_service(*, run=False):
    if not run:
        print("Shared service OFF. Set run=True only for an additional paid comparison.")
        return None
    require_paid_stage("nebius")
    require_paid_stage("anthropic")
    scoped_retriever = open_index(
        index_path, config, scoped_corpus, pages=page_repo, embeddings=backend, tokenizer=tokenizer
    )
    try:
        service = BaselineApplicationService(
            config,
            scoped_retriever,
            counter,
            generator,
            page_repo,
            JsonlTraceRecorder(settings.trace_root),
        )
        result = service.answer(request)
        print("Trace:", service.last_trace_id)
        pprint(result.model_dump(mode="json"))
        return result
    finally:
        scoped_retriever.store.client.close()


run_shared_service(run=False)

## 20. Finish or experiment again

Try changing the question first, then retrieval `TOP_K`/`CANDIDATE_COUNT`, then chunk
size/overlap. Change one thing at a time and compare retained evidence and citations.
Respect the validated caps. For config/document changes restart the kernel and rerun
setup; new text/chunk boundaries may require new embeddings. Edit small pure functions
locally to explore behavior, rerun dependent cells, and keep production changes separate
until reviewed. A new model/scenario needs a deliberate design and fresh verified rates.

When finished, save a receipt if desired, then explicitly close clients below or shut
down the kernel. Cleanup is off during Run All so you can call the stages afterward.
This notebook does not complete Phase 2's outstanding full-corpus acceptance gate.

In [ ]:
def close_resources(*, run=False):
    global retriever, embedding_http, chat, backend, counter, generator
    if not run:
        print("Cleanup OFF. Set run=True when finished, or shut down the kernel.")
        return
    if retriever is not None:
        retriever.store.client.close()
        retriever = None
    if embedding_http is not None:
        embedding_http.close()
        embedding_http = None
    if chat is not None:
        chat._client.close()
        chat = None
    backend = counter = generator = None


close_resources(run=False)